In [ ]:
import sys
!{sys.executable} -m pip install -U spacy
!{sys.executable} -m spacy download fr_core_news_sm

In [ ]:
!pip install tensorflow


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py", line 470, in _make_request
    self._validate_conn(conn)
    ~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py", line 1125, in _validate_conn
    conn.connect()
    ~~~~~~~~~~~~^^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connection.py", line 878, in connect
    wrapped_socket, is_verified = _ssl_wrap_socket_and_match_hostname(
                                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        sock=sock,
        ^^^^^^^^^^
    ...<14 lines>...
        assert_fingerprint=self.assert_fingerprint,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python31

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
!pip install spacy
!python -m spacy download fr_core_news_sm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py", line 470, in _make_request
    self._validate_conn(conn)
    ~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py", line 1125, in _validate_conn
    conn.connect()
    ~~~~~~~~~~~~^^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python314\site-packages\urllib3\connection.py", line 878, in connect
    wrapped_socket, is_verified = _ssl_wrap_socket_and_match_hostname(
                                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        sock=sock,
        ^^^^^^^^^^
    ...<14 lines>...
        assert_fingerprint=self.assert_fingerprint,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\jhaifa\AppData\Roaming\Python\Python31

## 1. Configuration

In [9]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)
import joblib
import spacy

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks
tf.get_logger().setLevel("ERROR")

### Prétraitement linguistique : spaCy

Les mots-outils français utilisés par la vectorisation TF-IDF viennent de **spaCy** (`fr_core_news_sm`) plutôt que d'une liste codée à la main : une liste vérifiée et maintenue par la communauté plutôt qu'une liste artisanale forcément incomplète. Un comparatif rapide (texte brut vs texte lemmatisé avec spaCy) a été mené en amont de ce notebook : la lemmatisation n'apportait pas de gain sur ce corpus (0.90 contre 0.92 de macro-F1 en validation croisée rapide) et complique l'export navigateur puisqu'aucun lemmatiseur léger n'existe côté JavaScript pour le français. Le texte brut (minuscule + stopwords spaCy) est donc conservé.

In [14]:
_NLP = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
SPACY_STOPWORDS = sorted(_NLP.Defaults.stop_words)
print(f"{len(SPACY_STOPWORDS)} mots-outils français chargés depuis spaCy (fr_core_news_sm).")

OSError: [E050] Can't find model 'fr_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

## 2. Chargement du dataset

Le corpus (`corpus.csv`, colonnes `text,label`) a été **rédigé à la main**, message par message : formulations courtes et longues, registre formel et informel, fautes et abréviations réalistes.

In [2]:
def load_corpus(path=DATA_CSV):
    df = pd.read_csv(path)
    assert "text" in df.columns and "label" in df.columns, (
        f"Le CSV doit contenir les colonnes 'text' et 'label' (trouvé : {list(df.columns)})"
    )
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    return df[df["text"].str.len() > 0].reset_index(drop=True)


def get_classes(df):
    # Ordre canonique des classes pour tout le pipeline : déduit des données
    # elles-memes (ordre alphabetique des labels presents), jamais d'une
    # liste codee en dur -- un autre CSV donnerait automatiquement un autre
    # ensemble de classes, sans toucher au code.
    return sorted(df["label"].unique().tolist())


df = load_corpus()
CLASSES = get_classes(df)
print(f"{len(df)} messages chargés, {len(CLASSES)} classes détectées automatiquement :")
print(CLASSES)
df.sample(8, random_state=3)

NameError: name 'DATA_CSV' is not defined

## 3. Analyse du dataset

Le sujet impose un jeu de classes déséquilibré et prévient que le choix des métriques doit en tenir compte. On vérifie ici l'ampleur réelle du déséquilibre, ainsi que la variété des longueurs de messages (le corpus contient volontairement des messages très courts, façon texto, et des messages longs, façon email détaillé).

In [ ]:
counts = df["label"].value_counts().reindex(CLASSES)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(counts.index, counts.values, color="#4C72B0")
ax.set_ylabel("Nombre de messages")
ax.set_title("Répartition des messages par service (corpus rédigé à la main)")
plt.xticks(rotation=35, ha="right")
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2, str(int(b.get_height())), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Classe majoritaire / minoritaire : {counts.max()} / {counts.min()} = ratio {counts.max()/counts.min():.1f}x")

In [ ]:
n_mots = df["text"].str.split().apply(len)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(n_mots, bins=20, color="#55A868", edgecolor="white")
ax.set_xlabel("Nombre de mots par message")
ax.set_ylabel("Nombre de messages")
ax.set_title("Distribution de la longueur des messages (courts et longs)")
plt.tight_layout()
plt.show()
print(n_mots.describe().round(1))
print("\nExemple de message court :", df.loc[n_mots.idxmin(), "text"])
print("Exemple de message long   :", df.loc[n_mots.idxmax(), "text"][:200], "...")

## 4. Préparation des données

Chaque famille de modèle a son propre prétraitement  :

| Famille | Prétraitement |
|---|---|
| TF-IDF | minuscule, stopwords spaCy, n-grammes (1,2), pondération sublinéaire |
| BiLSTM | tokenizer Keras (indices de mots), padding/troncature |
| CamemBERT / DistilCamemBERT | tokenizer sous-mots du modèle pré-entraîné |

Pour que la comparaison reste équitable, **aucune donnée de validation ou de test n'est jamais ré-échantillonnée pour équilibrer les classes** : le déséquilibre observé en section 3 est conservé tel quel partout, et c'est aux modèles (via `class_weight`, pondération de la loss, etc.) de s'y adapter — pas aux données d'être truquées.

In [ ]:
def align_proba(proba, fitted_classes, target_classes):
    # Réordonne les colonnes de proba pour correspondre exactement à l'ordre
    # de `target_classes`, quel que soit l'ordre interne du modèle.
    idx = [list(fitted_classes).index(c) for c in target_classes]
    return proba[:, idx]

## 5. Split développement / test

80% développement (utilisé pour la validation croisée et la sélection du modèle) / 20% test final, . Split stratifié pour que même les classes les plus rares restent représentées dans les deux ensembles.

In [ ]:
def split_dev_test(df, test_size=0.2, seed=SEED):
    dev_df, test_df = train_test_split(df, test_size=test_size, random_state=seed, stratify=df["label"])
    return dev_df.reset_index(drop=True), test_df.reset_index(drop=True)


dev_df, test_df = split_dev_test(df)
print(f"Développement : {len(dev_df)} messages   |   Test final (de côté) : {len(test_df)} messages")
print("\nRépartition (développement) :")
print(dev_df["label"].value_counts().reindex(CLASSES))
print("\nRépartition (test final) :")
print(test_df["label"].value_counts().reindex(CLASSES))

## 6. TF-IDF + Logistic Regression

La famille TF-IDF est déclinée en **5 candidats** (LogisticRegression, LinearSVC calibré, MultinomialNB, ComplementNB, RandomForest), tous construits avec exactement la même vectorisation. Chacun expose l'interface générique de la section 4. `LogisticRegression` sert de démonstration illustrative ci-dessous.

In [ ]:
def make_tfidf_pipeline(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=False, stop_words=SPACY_STOPWORDS,
                                   ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", clf),
    ])


def tfidf_build_and_fit(clf_factory):
    def build_and_fit(train_texts, train_labels, classes):
        pipe = make_tfidf_pipeline(clf_factory())
        pipe.fit(train_texts.str.lower(), train_labels)
        return pipe
    return build_and_fit


def tfidf_predict_proba(fitted, texts, classes):
    proba = fitted.predict_proba(texts.str.lower())
    return align_proba(proba, fitted.classes_, classes)


def tfidf_save(fitted, out_dir, name):
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{name}.joblib")
    joblib.dump(fitted, path)
    return {"model": path}


def tfidf_size(fitted):
    import io
    buf = io.BytesIO()
    joblib.dump(fitted, buf)
    return buf.tell()


def make_tfidf_candidate(name, clf_factory):
    safe = name.replace(" ", "_").replace("(", "").replace(")", "")
    return {
        "name": name, "family": "tfidf", "runnable": True, "reason_not_runnable": None,
        "build_and_fit": tfidf_build_and_fit(clf_factory),
        "predict_proba": lambda fitted, texts, classes: tfidf_predict_proba(fitted, texts, classes),
        "save": lambda fitted, out_dir: tfidf_save(fitted, out_dir, safe),
        "model_size_bytes": tfidf_size,
    }


TFIDF_CANDIDATES = [
    make_tfidf_candidate("TF-IDF + LogisticRegression",
                          lambda: LogisticRegression(max_iter=3000, class_weight="balanced", C=5.0, random_state=SEED)),
    make_tfidf_candidate("TF-IDF + LinearSVC (calibré)",
                          lambda: CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0, random_state=SEED), cv=3)),
    make_tfidf_candidate("TF-IDF + MultinomialNB", lambda: MultinomialNB(alpha=0.3)),
    make_tfidf_candidate("TF-IDF + ComplementNB", lambda: ComplementNB(alpha=0.3)),
    make_tfidf_candidate("TF-IDF + RandomForest",
                          lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1)),
]

# Démonstration illustrative (LogisticRegression) sur le split dev/test simple
demo = TFIDF_CANDIDATES[0]
t0 = time.time()
fitted_demo = demo["build_and_fit"](dev_df["text"], dev_df["label"], CLASSES)
print(f"Entraînement TF-IDF + LogisticRegression : {time.time()-t0:.2f}s")
proba_demo = demo["predict_proba"](fitted_demo, test_df["text"], CLASSES)
preds_demo = [CLASSES[i] for i in proba_demo.argmax(axis=1)]
print(f"Accuracy (démonstration, test final) : {accuracy_score(test_df['label'], preds_demo):.3f}")
print(f"Macro-F1 (démonstration, test final) : {f1_score(test_df['label'], preds_demo, average='macro'):.3f}")
print(f"Taille du modèle : {demo['model_size_bytes'](fitted_demo)/1024:.0f} Ko")